In [25]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

# 📦 Загрузка и предобработка датасета
behaviors = pd.read_csv("MINDsmall_train/behaviors.tsv", sep="\t", header=None,
                         names=["ImpressionID", "UserID", "Time", "History", "Impressions"])
behaviors = behaviors.head(2000)
# Преобразуем пользователей и новости в индексы
user_encoder = LabelEncoder()
news_encoder = LabelEncoder()

behaviors["UserID"] = user_encoder.fit_transform(behaviors["UserID"])
all_news = pd.read_csv("MINDsmall_train/news.tsv", sep="\t", header=None, names=["NewsID", "Category", "SubCategory", "Title", "Abstract", "URL", "TitleEntities", "AbstractEntities"])
all_news["NewsID"] = news_encoder.fit_transform(all_news["NewsID"])

# Создаём словарь для ускорения кодирования
news_id_to_idx = {news_id: idx for idx, news_id in enumerate(news_encoder.classes_)}

# Формируем данные
data = []
for _, row in behaviors.iterrows():
    user_id = row["UserID"]
    impressions = row["Impressions"].split()
    for imp in impressions:
        news_id, label = imp.split("-")
        encoded_news_id = news_id_to_idx[news_id]  # Быстрое преобразование
        data.append((int(user_id), int(encoded_news_id), int(label)))

# 🗂️ Dataset & DataLoader
class NCFDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        user, news, label = self.data[idx]
        return torch.tensor(user), torch.tensor(news), torch.tensor(label, dtype=torch.float32)

dataset = NCFDataset(data)
dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)

# 🔧 NCF-модель
class NCF(nn.Module):
    def __init__(self, num_users, num_items, emb_size=64, hidden_size=128):
        super(NCF, self).__init__()
        self.user_emb = nn.Embedding(num_users, emb_size)
        self.item_emb = nn.Embedding(num_items, emb_size)
        self.mlp = nn.Sequential(
            nn.Linear(emb_size*2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    
    def forward(self, user, item):
        u = self.user_emb(user)
        i = self.item_emb(item)
        x = torch.cat([u, i], dim=1)
        out = self.mlp(x).squeeze(1)
        return out

num_users = len(user_encoder.classes_)
num_items = len(news_encoder.classes_)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = NCF(num_users, num_items).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 🏋️ Обучение
for epoch in range(100):
    model.train()
    total_loss = 0
    for user, news, label in dataloader:
        user, news, label = user.to(device), news.to(device), label.to(device)
        optimizer.zero_grad()
        logits = model(user, news)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

print("Обучение завершено! 🎉")
torch.save(model.state_dict(), "ncf_model.pth")
print("Модель сохранена в ncf_model.pth.")


Epoch 1, Loss: 0.2706
Epoch 2, Loss: 0.1617
Epoch 3, Loss: 0.1563
Epoch 4, Loss: 0.1521
Epoch 5, Loss: 0.1479
Epoch 6, Loss: 0.1444
Epoch 7, Loss: 0.1406
Epoch 8, Loss: 0.1371
Epoch 9, Loss: 0.1338
Epoch 10, Loss: 0.1302
Epoch 11, Loss: 0.1269
Epoch 12, Loss: 0.1231
Epoch 13, Loss: 0.1194
Epoch 14, Loss: 0.1149
Epoch 15, Loss: 0.1103
Epoch 16, Loss: 0.1053
Epoch 17, Loss: 0.0998
Epoch 18, Loss: 0.0940
Epoch 19, Loss: 0.0879
Epoch 20, Loss: 0.0815
Epoch 21, Loss: 0.0751
Epoch 22, Loss: 0.0685
Epoch 23, Loss: 0.0618
Epoch 24, Loss: 0.0559
Epoch 25, Loss: 0.0496
Epoch 26, Loss: 0.0443
Epoch 27, Loss: 0.0390
Epoch 28, Loss: 0.0343
Epoch 29, Loss: 0.0301
Epoch 30, Loss: 0.0262
Epoch 31, Loss: 0.0228
Epoch 32, Loss: 0.0198
Epoch 33, Loss: 0.0171
Epoch 34, Loss: 0.0149
Epoch 35, Loss: 0.0129
Epoch 36, Loss: 0.0112
Epoch 37, Loss: 0.0098
Epoch 38, Loss: 0.0086
Epoch 39, Loss: 0.0075
Epoch 40, Loss: 0.0067
Epoch 41, Loss: 0.0060
Epoch 42, Loss: 0.0054
Epoch 43, Loss: 0.0048
Epoch 44, Loss: 0.00

ValueError: invalid literal for int() with base 10: 'N55689'

In [35]:
# Предположим, у тебя есть пользователь
user_str_id = "U13740"  # пример строкового ID

# И список новостей, которые ты хочешь проверить
news_str_ids = [
    "N55189", "N42782", "N34694", "N45794", "N18445", "N63302", "N10414", "N19347", "N31801",
    "N31739", "N6072", "N63045", "N23979", "N35656", "N43353", "N8129", "N1569", "N17686", "N13008", "N21623",
    "N6233", "N14340", "N48031", "N62285", "N44383", "N23061", "N16290", "N6244", "N45099", "N58715", "N59049",
    "N7023", "N50528", "N42704", "N46082", "N8275", "N15710", "N59026", "N8429", "N30867", "N56514", "N19709",
    "N31402", "N31741", "N54889", "N9798", "N62612", "N2663", "N16617", "N6087", "N13231", "N63317", "N61388",
    "N59359", "N51163", "N30698", "N34567", "N54225", "N32852", "N55833", "N64467", "N3142", "N13912", "N29802",
    "N44462", "N29948", "N4486", "N5398", "N14761", "N47020", "N65112", "N31699", "N37159", "N61101", "N14761",
    "N3433", "N10438", "N61355", "N21164", "N22976", "N2511", "N48390", "N58224", "N48742", "N35458", "N24611",
    "N37509", "N21773", "N41011", "N19041", "N25785",
    "N10732", "N25792", "N7563", "N21087", "N41087", "N5445", "N60384", "N46616", "N52500", "N33164", "N47289",
    "N24233", "N62058", "N26378", "N49475", "N18870",
    "N45729", "N2203", "N871", "N53880", "N41375", "N43142", "N33013", "N29757", "N31825", "N51891",
    "N10078", "N56514", "N14904", "N33740",
    "N39074", "N14343", "N32607", "N32320", "N22007", "N442", "N19001", "N24294", "N51188", "N22772", "N51188",
    "N12603", "N8275", "N19741", "N6695", "N35820", "N30531", "N15545", "N27529", "N62703", "N59426", "N15414",
    "N54827", "N21395", "N39941", "N10824", "N42512", "N58521", "N62846", "N14385", "N47020", "N2142", "N17099",
    "N47020", "N11804", "N52121",
    "N8419", "N15771", "N1431", "N5888", "N18663", "N24123", "N22130", "N20286", "N32095", "N46868", "N55310",
    "N31931", "N34399", "N42526", "N64562", "N12194", "N23887", "N56541", "N59704", "N30531", "N60388", "N8569",
    "N38562", "N47791", "N157", "N306", "N30160", "N41797", "N47482", "N2606", "N20886", "N47054", "N64631",
    "N3933", "N40509",
    "N47438", "N20950", "N21317", "N5469"
]


# Кодируем их
user_id = user_encoder.transform([user_str_id])[0]  # один пользователь
news_ids = news_encoder.transform(news_str_ids)     # массив новостей

# Преобразуем в тензоры
user_tensor = torch.tensor([user_id] * len(news_ids)).to(device)  # повторяем user_id для каждой новости
news_tensor = torch.tensor(news_ids).to(device)

# Получаем предсказания
with torch.no_grad():
    logits = model(user_tensor, news_tensor)
    probabilities = torch.sigmoid(logits).cpu().numpy()

# Выводим результаты
for news_str_id, prob in zip(news_str_ids, probabilities):
    print(f"Вероятность клика пользователя {user_str_id} на новость {news_str_id}: {prob:.4f}")

Вероятность клика пользователя U13740 на новость N55189: 0.0000
Вероятность клика пользователя U13740 на новость N42782: 0.0244
Вероятность клика пользователя U13740 на новость N34694: 0.0000
Вероятность клика пользователя U13740 на новость N45794: 0.0479
Вероятность клика пользователя U13740 на новость N18445: 0.0000
Вероятность клика пользователя U13740 на новость N63302: 0.0000
Вероятность клика пользователя U13740 на новость N10414: 0.0085
Вероятность клика пользователя U13740 на новость N19347: 0.0153
Вероятность клика пользователя U13740 на новость N31801: 0.0000
Вероятность клика пользователя U13740 на новость N31739: 0.0000
Вероятность клика пользователя U13740 на новость N6072: 0.1585
Вероятность клика пользователя U13740 на новость N63045: 0.0000
Вероятность клика пользователя U13740 на новость N23979: 0.0000
Вероятность клика пользователя U13740 на новость N35656: 0.0000
Вероятность клика пользователя U13740 на новость N43353: 0.0000
Вероятность клика пользователя U13740 на 